# 1) EE-1.1 Circuits from zero — Exercises

**Goals:** power vs. energy units, I²R transmission losses, nodal analysis by hand and in code, building a conductance matrix automatically, and understanding why a reference node is needed.

Units: W/kW/MW for power, Wh/kWh/MWh/TWh for energy, V, A, Ω, S. Do the hand calculations (task 6) on paper first — the notebook only checks your final numbers.

**Submit** your notebook (and a photo of the paper work for task 6) for review before opening the solution in `solutions/`.

In [1]:
import math
import numpy as np

### Warm-ups

**1. Kettle energy**

A 2 kW kettle runs for 3 minutes. Return the energy used in kWh.

In [2]:
def kettle_energy_kWh(power_kW: float, minutes: float) -> float:
    """Energy (kWh) used by a device of power_kW running for the given minutes."""
    ### BEGIN SOLUTION
    return power_kW * minutes / 60
    ### END SOLUTION

In [3]:
assert math.isclose(kettle_energy_kWh(2, 3), 0.1), "kW x hours = kWh; 3 minutes is 0.05 h"

**2. Average demand from annual consumption**

Croatia consumes roughly 18 TWh of electricity per year. Return the *average* demand in MW (use 8,760 hours per year).

In [4]:
def average_demand_MW(annual_TWh: float, hours_per_year: float = 8760) -> float:
    """Average power (MW) that delivers annual_TWh of energy over a year."""
    ### BEGIN SOLUTION
    return annual_TWh * 1e6 / hours_per_year
    ### END SOLUTION

In [5]:
assert math.isclose(average_demand_MW(18), 18e6 / 8760), "1 TWh = 1,000,000 MWh"
print(f"Average demand: {average_demand_MW(18):,.0f} MW")

Average demand: 2,055 MW


**3. Spot the unit error**

A model input file says: lignite ramp limit **"200 MW/h"**, battery capacity **"50 MW"**. Which, if either, is wrong or ambiguous, and why?

<!-- BEGIN SOLUTION -->
"200 MW/h" is fine: a ramp limit is a rate of change of *power*, so MW per hour is the right unit. "50 MW" is wrong or at best ambiguous for a battery *capacity*: MW describes the power rating (how fast it can charge or discharge), while the storage capacity is an *energy* and must be in MWh. A battery needs both numbers (e.g. 50 MW / 200 MWh).
<!-- END SOLUTION -->

### Core

**4. Transmission losses**

Deliver 500 MW over a line with R = 5 Ω (DC simplification). Write `line_loss_MW`, then `required_voltage_kV`: the lowest voltage that keeps the loss share at or below `max_loss_share` (e.g. 0.01 for 1 %). Derive the formula for the second function on paper first.

In [6]:
def line_loss_MW(power_MW: float, voltage_kV: float, resistance_ohm: float) -> float:
    """I²R loss (MW) when delivering power_MW at voltage_kV over resistance_ohm."""
    ### BEGIN SOLUTION
    current_A = power_MW * 1e6 / (voltage_kV * 1e3)
    return current_A**2 * resistance_ohm / 1e6
    ### END SOLUTION


def required_voltage_kV(power_MW: float, resistance_ohm: float, max_loss_share: float) -> float:
    """Lowest voltage (kV) so that loss / power <= max_loss_share."""
    ### BEGIN SOLUTION
    # loss/P = P R / V^2 <= s  ->  V >= sqrt(P R / s)
    return math.sqrt(power_MW * 1e6 * resistance_ohm / max_loss_share) / 1e3
    ### END SOLUTION

In [7]:
assert math.isclose(line_loss_MW(500, 220, 5), (500e6 / 220e3) ** 2 * 5 / 1e6)
assert math.isclose(line_loss_MW(500, 400, 5), 7.8125), "at 400 kV the current is 1,250 A"
assert math.isclose(required_voltage_kV(500, 5, 0.01), 500.0)
for kv in [220, 400]:
    loss = line_loss_MW(500, kv, 5)
    print(f"{kv} kV: loss {loss:.2f} MW ({100 * loss / 500:.2f} %)")

220 kV: loss 25.83 MW (5.17 %)
400 kV: loss 7.81 MW (1.56 %)


**5. Interpretation**

Losses fall with the square of the voltage. Give at least two engineering reasons why grids don't simply run at 1,000 kV everywhere.

<!-- BEGIN SOLUTION -->
Higher voltage needs more insulation and larger clearances, so towers, insulators, substations and transformers become much bigger and more expensive. Corona discharge (energy lost to ionising the air around conductors, plus noise and interference) grows with voltage. Every connection to the lower-voltage grid needs costly transformation. So very high voltages only pay off for moving large amounts of power over long distances; regional networks use lower voltages where the extra cost would never be recovered by lower losses.
<!-- END SOLUTION -->

**6. Nodal analysis by hand**

Network: branches 1–2 (2 Ω), 2–3 (2 Ω), 1–3 (4 Ω), 3–0 (2 Ω); node 0 is the reference; a source injects 5 A into node 1.

On paper: sketch the circuit, write KCL at nodes 1, 2, 3, and solve. Then enter your conductance matrix and solve it here.

In [8]:
def e6_conductance_matrix() -> np.ndarray:
    """3x3 conductance matrix (S) for nodes 1, 2, 3 of the task-6 network."""
    ### BEGIN SOLUTION
    return np.array([[0.5 + 0.25, -0.5, -0.25],
                     [-0.5, 0.5 + 0.5, -0.5],
                     [-0.25, -0.5, 0.25 + 0.5 + 0.5]])
    ### END SOLUTION

In [9]:
G6 = e6_conductance_matrix()
assert np.allclose(G6, G6.T), "a conductance matrix is symmetric"
V6 = np.linalg.solve(G6, np.array([5.0, 0.0, 0.0]))
assert np.allclose(V6, [20.0, 15.0, 10.0]), "check the diagonal: sum of all conductances at the node"
V6

array([20., 15., 10.])

**7. Build the conductance matrix automatically**

Write `solve_dc_network`. It takes a branch list `[(from_node, to_node, R_ohm), ...]` where node 0 is the reference, the injected currents for nodes 1…n, and the number of non-reference nodes. It must **build G from the branch list** (don't type the matrix), solve for node voltages, compute every branch current, and **assert KCL at every node and the energy balance** before returning.

In [10]:
def solve_dc_network(branches: list[tuple[int, int, float]],
                     injections_A: list[float],
                     n_nodes: int) -> tuple[np.ndarray, dict]:
    """Solve a DC resistive network with nodal analysis.

    branches: (from_node, to_node, resistance_ohm); node 0 is the reference.
    injections_A: current injected at nodes 1..n_nodes (A).
    Returns (node voltages for nodes 1..n in V, {(from, to): current_A}).
    """
    ### BEGIN SOLUTION
    G = np.zeros((n_nodes, n_nodes))
    for a, b, r in branches:
        g = 1.0 / r
        for node in (a, b):
            if node != 0:
                G[node - 1, node - 1] += g
        if a != 0 and b != 0:
            G[a - 1, b - 1] -= g
            G[b - 1, a - 1] -= g
    inj = np.asarray(injections_A, dtype=float)
    V = np.linalg.solve(G, inj)
    v_full = np.concatenate([[0.0], V])  # index = node number
    currents = {(a, b): (v_full[a] - v_full[b]) / r for a, b, r in branches}

    # KCL: injection = sum of currents leaving the node
    for node in range(1, n_nodes + 1):
        leaving = sum(i if a == node else -i for (a, b), i in currents.items() if node in (a, b))
        assert np.isclose(leaving, inj[node - 1]), f"KCL violated at node {node}"
    # energy balance: power injected = power dissipated
    p_in = float(V @ inj)
    p_heat = sum(i**2 * r for (a, b, r), i in zip(branches, currents.values()))
    assert np.isclose(p_in, p_heat), "energy balance violated"
    return V, currents
    ### END SOLUTION

In [11]:
V_ex, I_ex = solve_dc_network([(1, 0, 4.0), (1, 2, 2.0), (2, 0, 4.0)], [3.0, 0.0], 2)
assert np.allclose(V_ex, [7.2, 4.8]), "worked example from the lecture"

V6b, I6b = solve_dc_network([(1, 2, 2.0), (2, 3, 2.0), (1, 3, 4.0), (3, 0, 2.0)], [5.0, 0.0, 0.0], 3)
assert np.allclose(V6b, [20.0, 15.0, 10.0])
assert np.isclose(I6b[(3, 0)], 5.0), "all 5 A must leave through the only path to the reference"
I6b

{(1, 2): np.float64(2.5),
 (2, 3): np.float64(2.499999999999999),
 (1, 3): np.float64(2.4999999999999996),
 (3, 0): np.float64(4.999999999999997)}

### Challenge

**8. Why a reference node?**

Build the conductance matrix of the task-6 network **including node 0 as an unknown** (4×4). Return it, then check its rank. Explain below why `np.linalg.solve` cannot work with it — mathematically (look at the row sums) and physically.

In [12]:
def full_conductance_matrix(branches: list[tuple[int, int, float]], n_nodes_incl_ref: int) -> np.ndarray:
    """Conductance matrix including the reference node as an ordinary node."""
    ### BEGIN SOLUTION
    G = np.zeros((n_nodes_incl_ref, n_nodes_incl_ref))
    for a, b, r in branches:
        g = 1.0 / r
        G[a, a] += g
        G[b, b] += g
        G[a, b] -= g
        G[b, a] -= g
    return G
    ### END SOLUTION

In [13]:
G_full = full_conductance_matrix([(1, 2, 2.0), (2, 3, 2.0), (1, 3, 4.0), (3, 0, 2.0)], 4)
assert G_full.shape == (4, 4)
assert np.allclose(G_full.sum(axis=1), 0), "every row should sum to zero"
assert np.linalg.matrix_rank(G_full) == 3, "one equation is redundant"
print("row sums:", G_full.sum(axis=1), "| rank:", np.linalg.matrix_rank(G_full))

row sums: [0. 0. 0. 0.] | rank: 3


<!-- BEGIN SOLUTION -->
Every row sums to zero, so the columns are linearly dependent: the vector of all ones is in the null space (G · 1 = 0). The matrix is singular (rank 3, not 4) and has no inverse, so there is no unique solution. Physically: only voltage *differences* drive currents. If V solves the network, so does V + c for any constant c — adding the same voltage everywhere changes no current. Fixing one node at 0 V (the reference, in power systems the slack bus) removes that freedom and makes the solution unique.
<!-- END SOLUTION -->

**9. How does the current split?**

In the task-6 network, current can flow from node 1 to node 3 directly or via node 2. Which path carries more, and which of Kirchhoff's laws decides the split?

<!-- BEGIN SOLUTION -->
They carry exactly the same: 2.5 A each. Both paths connect the same two nodes, so by **KVL** they see the same voltage difference, V₁ − V₃ = 10 V. The direct path has 4 Ω and the path via node 2 has 2 + 2 = 4 Ω, so each carries 10/4 = 2.5 A; KCL then adds them to the 5 A leaving node 3. In general current splits in inverse proportion to resistance — the network, not a planner, decides the flows. That is the key difference between a real grid and a transport model.
<!-- END SOLUTION -->